In [94]:
import pygad
import numpy as np
import itertools
import math

In [95]:
A = [7, 8, 18]
B = [1,2,2,4,5,5,5,9]
C = [1,2,2,2,2,3,4,5,5,7]
m = len(A)
n = len(B)
target = sorted(C)
print(f"A:{len(A)} B: {len(B)} C: {len(C)}")

A:3 B: 8 C: 10


In [96]:
def rank_perm(p):
    n = len(p)
    rank = 0
    for i in range(n):
        smaller = sum(p[j] < p[i] for j in range(i+1, n))
        rank += smaller * math.factorial(n - i - 1)
    return rank

In [97]:
def unrank_perm(n, rank):
    import math

    elements = list(range(n))
    perm = []

    for i in range(n-1, -1, -1):
        fact = math.factorial(i)
        idx = rank // fact
        rank %= fact
        perm.append(elements.pop(idx))

    return perm

# Permutations of A

In [98]:
p = [1,2,0]
r = rank_perm(p)
print(f"Rank {r} ")

Rank 3 


In [99]:
perm_indices = unrank_perm(3, 3)
print(perm_indices)

[1, 2, 0]


In [100]:
perm_values = [A[i] for i in perm_indices]
print(perm_values)

[8, 18, 7]


# Permutations of B

In [101]:
perm_indices = unrank_perm(8, 3)
print(perm_indices)

[0, 1, 2, 3, 4, 6, 7, 5]


In [102]:
perm_values = [B[i] for i in perm_indices]
print(perm_values)

[1, 2, 2, 4, 5, 5, 9, 5]


# GA Implementation using ranks

In [103]:
no_ranks_a = math.factorial(len(A))
no_ranks_b = math.factorial(len(B))
print(f"no_ranks_a: {no_ranks_a}, no_ranks_b: {no_ranks_b}")

no_ranks_a: 6, no_ranks_b: 40320


In [104]:
no_bits_rank_a = math.ceil(math.log2(no_ranks_a))
no_bits_rank_b = math.ceil(math.log2(no_ranks_b))
print(f"no_bits_rank_a: {no_bits_rank_a}, no_bits_rank_b: {no_bits_rank_b}")

no_bits_rank_a: 3, no_bits_rank_b: 16


In [105]:
total_bits = no_bits_rank_a + no_bits_rank_b
print(f"total_bits: {total_bits}")

total_bits: 19


# GA Implementation

In [106]:
def cumulative_sum(arr):
    return np.cumsum(arr).tolist()

In [107]:
def step_difference(arr):
    return [arr[0]] + [arr[i] - arr[i-1] for i in range(1, len(arr))]

In [108]:
gene_space = [0,1]

In [114]:
def decode_solution(solution):
    rank_a_bits = solution[:no_bits_rank_a]
    rank_b_bits = solution[no_bits_rank_a:]
    rank_a = int(''.join(map(str, rank_a_bits)), 2)
    rank_b = int(''.join(map(str, rank_b_bits)), 2)
    if rank_a >= no_ranks_a or rank_a < 0:
        rank_a = 0
    if rank_b >= no_ranks_b or rank_b < 0:
        rank_b = 0
    #print(f"rank_a: {rank_a}, rank_b: {rank_b}")
    perm_a = unrank_perm(m, rank_a)
    perm_b = unrank_perm(n, rank_b)
    return perm_a,perm_b

In [115]:
def fitness_func(ga, solution, solution_idx) -> float:
    # chromosome encoding: [ u | ν ]
    mu,nu = decode_solution(solution)



    # Check if valid permutations (exact set match)
    if set(mu) != set(range(m)) or set(nu) != set(range(n)):
        return 0.0  # invalid permutation

    A_perm = [A[i] for i in mu]
    B_perm = [B[i] for i in nu]
    C_mod = list(target)

    k = len(C_mod)
    expected_k = m + n - 1

    # --- REPAIR SEQUENCE LOGIC ---
    if k < expected_k:
        # Rule 1: Add 0-length fragments to target C
        delta = expected_k - k
        C_mod.extend([0] * delta)

    elif k > expected_k:
        # Rule 2: Add 0-length fragments to the shorter sequence among A and B
        delta = k - expected_k
        if m <= n:
            A_perm.extend([0] * delta)
        else:
            B_perm.extend([0] * delta)

    A_cum = cumulative_sum(A_perm)
    B_cum = cumulative_sum(B_perm)

    merged = sorted(A_cum + B_cum)

    steps = step_difference(merged)

    steps_sorted = sorted(set(steps))



    error = sum(abs(s - t) for s, t in zip(steps_sorted, target))
    fitness = 1.0 / (1.0 + error)
    #print(f"mu: {mu}; nu:{nu} -> fitness: {fitness}")
    return fitness

In [119]:
ga = pygad.GA(
    num_generations=1000,
    num_parents_mating=50,
    gene_type=int,
    fitness_func=fitness_func,
    sol_per_pop= 200,
    num_genes= total_bits,
    gene_space=gene_space,
    parent_selection_type="tournament",
    crossover_type="single_point",
    crossover_probability=0.75,
    mutation_type="random",
    mutation_probability=0.2,
    keep_parents=25,
    allow_duplicate_genes=True,

)

In [120]:
ga.run()

In [121]:
# Optional: Print best solution
solution, solution_fitness, solution_idx = ga.best_solution()
print(f"Best fitness: {solution_fitness}")
print(f"Best solution (binary): {solution}")
mu_bin, nu_bin = decode_solution(solution)
print(f"Decoded mu: {mu_bin}, nu: {nu_bin}")

Best fitness: 0.125
Best solution (binary): [0 0 1 0 0 1 1 1 0 1 0 1 0 0 1 1 0 0 1]
Decoded mu: [0, 2, 1], nu: [2, 7, 6, 0, 1, 3, 5, 4]
